# First CNN vs CNN Take 4

This notebook compares the original CNN baseline against **CNN Take 4**, which is the strongest CNN run in the project so far.

Goal:
- compare the first CNN with the final CNN Take 4 model
- explain the main architecture and training changes
- show whether accuracy improved and by how much

This is not a strict one-variable-only ablation. The improvement came from a combination of better architecture choices and a stronger training recipe.

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from SRC.cnn_model import SimpleCIFAR10CNN
from SRC.improved_cnn import DeeperCIFAR10CNN

BASELINE_SUMMARY_PATH = PROJECT_ROOT / "Data" / "cnn_baseline_outputs" / "run_20260325_130955" / "summary.json"
TAKE4_SUMMARY_PATH = PROJECT_ROOT / "Data" / "improved_cnn_outputs" / "cnn_take_4_20260326_234323" / "summary.json"
UPGRADED_SUMMARY_PATH = PROJECT_ROOT / "Data" / "improved_cnn_outputs" / "run_20260326_161003" / "summary.json"

with BASELINE_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    baseline = json.load(handle)

with TAKE4_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    take4 = json.load(handle)

with UPGRADED_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    upgraded = json.load(handle)

baseline["test_accuracy"], take4["test_accuracy"]

## 1. Headline Result

The main question is whether the final CNN Take 4 model beat the first CNN baseline on the held-out test set.

In [ ]:
absolute_gain = take4["test_accuracy"] - baseline["test_accuracy"]
relative_gain = absolute_gain / baseline["test_accuracy"]

headline = pd.DataFrame(
    [
        {
            "model": "First CNN baseline",
            "best_val_accuracy": baseline["best_val_accuracy"],
            "test_accuracy": baseline["test_accuracy"],
            "test_loss": baseline["test_loss"],
        },
        {
            "model": "CNN Take 4",
            "best_val_accuracy": take4["best_val_accuracy"],
            "test_accuracy": take4["test_accuracy"],
            "test_loss": take4["test_loss"],
        },
    ]
)

display(headline)
print(f"Absolute test accuracy gain: {absolute_gain:.4f}")
print(f"Relative gain over the baseline: {relative_gain:.2%}")

## 2. What Changed From the First CNN?

Some changes were architectural, and some were training-related. Together, they produced the final improvement.

In [ ]:
baseline_model = SimpleCIFAR10CNN(dropout=baseline["dropout"])
take4_model = DeeperCIFAR10CNN(
    channels=tuple(take4["channels"]),
    blocks_per_stage=tuple(take4["blocks_per_stage"]),
    pooling_type=take4["pooling_type"],
    dropout=take4["dropout"],
    classifier_hidden=take4["classifier_hidden"],
)

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

changes = pd.DataFrame(
    [
        {"setting": "epochs", "first_cnn": baseline["epochs"], "cnn_take_4": take4["epochs"], "comment": "Trained longer with a stronger schedule."},
        {"setting": "batch_size", "first_cnn": baseline["batch_size"], "cnn_take_4": take4["batch_size"], "comment": "Stayed the same at 256."},
        {"setting": "learning_rate", "first_cnn": baseline["learning_rate"], "cnn_take_4": take4["learning_rate"], "comment": "Lower and paired with cosine decay."},
        {"setting": "weight_decay", "first_cnn": baseline["weight_decay"], "cnn_take_4": take4["weight_decay"], "comment": "Stronger regularization for the larger model."},
        {"setting": "dropout", "first_cnn": baseline["dropout"], "cnn_take_4": take4["dropout"], "comment": "Reduced so more signal reaches the classifier."},
        {"setting": "optimizer", "first_cnn": "adam", "cnn_take_4": take4["optimizer"], "comment": "Changed from Adam to AdamW."},
        {"setting": "label_smoothing", "first_cnn": 0.0, "cnn_take_4": take4["label_smoothing"], "comment": "Helps reduce overconfidence."},
        {"setting": "augmentation", "first_cnn": baseline["augment"], "cnn_take_4": take4["augment"], "comment": "Take 4 uses a much stronger augmentation pipeline."},
        {"setting": "augmentation_policy", "first_cnn": "none", "cnn_take_4": take4["augmentation_policy"], "comment": "Take 4 combines basic spatial augmentation with extra color augmentation."},
        {"setting": "random_erasing_prob", "first_cnn": 0.0, "cnn_take_4": take4["random_erasing_prob"], "comment": "Randomly removes parts of training images."},
        {"setting": "color_jitter", "first_cnn": "none", "cnn_take_4": f"b={take4['color_jitter_brightness']}, c={take4['color_jitter_contrast']}, s={take4['color_jitter_saturation']}, h={take4['color_jitter_hue']}", "comment": "Makes the model less dependent on exact color conditions."},
        {"setting": "random_grayscale_prob", "first_cnn": 0.0, "cnn_take_4": take4["random_grayscale_prob"], "comment": "Reduces over-reliance on color only."},
        {"setting": "MixUp alpha", "first_cnn": 0.0, "cnn_take_4": take4["mixup_alpha"], "comment": "Mixes images and labels during training."},
        {"setting": "CutMix alpha", "first_cnn": 0.0, "cnn_take_4": take4["cutmix_alpha"], "comment": "Cuts and pastes image regions between samples."},
        {"setting": "scheduler", "first_cnn": "none", "cnn_take_4": take4["scheduler"], "comment": "Cosine schedule with warmup."},
        {"setting": "pooling_type", "first_cnn": "max", "cnn_take_4": take4["pooling_type"], "comment": "Take 4 uses learned stride downsampling."},
        {"setting": "channels", "first_cnn": "32, 64, 128", "cnn_take_4": ", ".join(str(value) for value in take4["channels"]), "comment": "Wider and deeper feature hierarchy."},
        {"setting": "blocks_per_stage", "first_cnn": "2, 2, 2", "cnn_take_4": ", ".join(str(value) for value in take4["blocks_per_stage"]), "comment": "More flexible stage design."},
        {"setting": "batch_norm", "first_cnn": "no", "cnn_take_4": "yes", "comment": "Each block uses Conv + BatchNorm + ReLU."},
        {"setting": "classifier_hidden", "first_cnn": "none", "cnn_take_4": take4["classifier_hidden"], "comment": "Take 4 has a larger classifier head."},
        {"setting": "trainable_parameters", "first_cnn": count_parameters(baseline_model), "cnn_take_4": count_parameters(take4_model), "comment": "The stronger model has much higher capacity."},
    ]
)

display(changes)

## 3. Progress Across Runs

The project did not jump from the first CNN directly to Take 4 in one step. There was a clear progression.

In [ ]:
progress = pd.DataFrame(
    [
        {"run": "First CNN baseline", "test_accuracy": baseline["test_accuracy"], "best_val_accuracy": baseline["best_val_accuracy"]},
        {"run": "Improved CNN", "test_accuracy": 0.7447, "best_val_accuracy": 0.7590},
        {"run": "Upgraded CNN", "test_accuracy": upgraded["test_accuracy"], "best_val_accuracy": upgraded["best_val_accuracy"]},
        {"run": "CNN Take 4", "test_accuracy": take4["test_accuracy"], "best_val_accuracy": take4["best_val_accuracy"]},
    ]
)

display(progress)

plt.figure(figsize=(9, 5))
plt.plot(progress["run"], progress["test_accuracy"], marker="o", linewidth=2)
plt.axhline(0.80, color="black", linestyle="--", linewidth=1, label="80% target")
plt.ylabel("Test Accuracy")
plt.ylim(0.5, 0.86)
plt.title("CNN Progress Over Time")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Training Curves: Baseline vs CNN Take 4

Comparing the histories shows how much more effectively Take 4 learned over time.

In [ ]:
baseline_history = pd.DataFrame(baseline["history"])
take4_history = pd.DataFrame(take4["history"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(baseline_history["epoch"], baseline_history["train_loss"], marker="o", label="Baseline train")
axes[0].plot(baseline_history["epoch"], baseline_history["val_loss"], marker="o", label="Baseline val")
axes[0].plot(take4_history["epoch"], take4_history["train_loss"], marker="o", label="Take 4 train")
axes[0].plot(take4_history["epoch"], take4_history["val_loss"], marker="o", label="Take 4 val")
axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(baseline_history["epoch"], baseline_history["train_accuracy"], marker="o", label="Baseline train")
axes[1].plot(baseline_history["epoch"], baseline_history["val_accuracy"], marker="o", label="Baseline val")
axes[1].plot(take4_history["epoch"], take4_history["train_accuracy"], marker="o", label="Take 4 train")
axes[1].plot(take4_history["epoch"], take4_history["val_accuracy"], marker="o", label="Take 4 val")
axes[1].axhline(baseline["test_accuracy"], color="tab:blue", linestyle="--", alpha=0.7, label="Baseline test")
axes[1].axhline(take4["test_accuracy"], color="tab:red", linestyle="--", alpha=0.7, label="Take 4 test")
axes[1].set_title("Accuracy by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Interpretation

What do these results suggest?

- The final CNN Take 4 clearly outperformed the first CNN on both validation and test accuracy.
- The gain came from a combination of architecture changes and a much stronger training recipe.
- Take 4 also crossed the `80%` test accuracy target, which the earlier runs had not reached.
- Learned stride downsampling and explicit color augmentation appear to have helped beyond the earlier upgraded CNN recipe.

Main conclusion: yes, the CNN improved substantially, and CNN Take 4 is the strongest result in the project so far.

In [ ]:
print(f"Baseline test accuracy : {baseline['test_accuracy']:.4f}")
print(f"CNN Take 4 accuracy    : {take4['test_accuracy']:.4f}")
print(f"Absolute improvement   : {absolute_gain:.4f}")
print(f"Relative improvement   : {relative_gain:.2%}")

if absolute_gain > 0:
    print("Conclusion: yes, CNN Take 4 achieved higher test accuracy than the first CNN.")
else:
    print("Conclusion: no, CNN Take 4 did not beat the first CNN on test accuracy.")